# Basic LLM: TinyAgent Live Build

This notebook builds an LLM adapter, parses structured endpoint responses, and connects an answer-only agent called TinyAgent to a trajectory log. The build runs fully offline using a scripted fixture, so no model download or local server is needed. The fixture is a lookup, not a model: it can prove the software path works, but it cannot prove anything about a real model's accuracy.

Source: Grootendorst and Alammar, *An Illustrated Guide to AI Agents*, Chapter 2.

## Step 1: Messages and a context budget

Roles identify who supplies each message. The system message shapes the intended behavior without changing any model weights. Before adding more input, the context budget reserves room for the output: 4096 - 3000 - 500 = 596 tokens remain. These are supplied counts for the exercise, not measurements from a real tokenizer. History, tool descriptions, retrieved text, and formatting overhead would all use more of that input space.

In [1]:
import json
from dataclasses import dataclass, field, asdict
import urllib.request

messages = [
    {"role": "system", "content": "Answer briefly and accurately."},
    {"role": "user", "content": "What is 2 + 2?"},
]
print(json.dumps(messages, indent=2))
context_limit, input_tokens, reserved_output = 4096, 3000, 500
print("Remaining budget:", context_limit - input_tokens - reserved_output)

[
  {
    "role": "system",
    "content": "Answer briefly and accurately."
  },
  {
    "role": "user",
    "content": "What is 2 + 2?"
  }
]
Remaining budget: 596


## Step 2: A structured response

A dataclass stores named fields with a readable representation. Optional fields may be absent (`None`). `default_factory=dict` gives each instance its own dictionary, so instances never share one. A response record only holds what came back; it is not evidence that its content is correct.

In [2]:
@dataclass
class Response:
    content: str = ""
    reasoning: str | None = None
    tool_call: dict | None = None
    metadata: dict = field(default_factory=dict)

example = Response(content="4")
print(example)
print(example.metadata)

Response(content='4', reasoning=None, tool_call=None, metadata={})
{}


## Step 3: Parse an endpoint response

Parsing is kept separate from networking so the boundary can be inspected. The result contains choices, each with a message. The parser handles absent usage and null content, and turns an empty choices list into a clear error. It keeps only the first choice and the first tool call. Extracting a tool call does not execute it; software still has to dispatch it.

In [3]:
def parse_response(data):
    choices = data.get("choices") or []
    if not choices:
        raise ValueError("Endpoint returned no choices")
    message = choices[0]["message"]
    calls = message.get("tool_calls") or []
    return Response(
        content=message.get("content") or "",
        reasoning=message.get("reasoning"),
        tool_call=calls[0] if calls else None,
        metadata={"model": data.get("model", "unknown"),
                  "usage": data.get("usage") or {}},
    )

sample = {"choices": [{"message": {"content": "4"}}]}
print(parse_response(sample))

Response(content='4', reasoning=None, tool_call=None, metadata={'model': 'unknown', 'usage': {}})


## Step 4: Transport and the LLM wrapper

HTTP gets one job and the wrapper gets another. `json.dumps` serializes a dictionary, `.encode` produces request bytes, and `urlopen` sends the HTTP request with a 60-second timeout so the program never waits forever. The wrapper builds the request body and delegates parsing to `parse_response`. Because the wrapper accepts an alternate transport callable, parsing can be tested without any network. No request happens just by defining these functions.

In [4]:
def http_transport(url, body):
    request = urllib.request.Request(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=60) as reply:
        return json.loads(reply.read().decode("utf-8"))

class LLM:
    def __init__(self, model, transport=http_transport,
                 base_url="http://localhost:11434/v1"):
        self.model = model
        self.transport = transport
        self.base_url = base_url.rstrip("/")

    def generate(self, messages):
        body = {"model": self.model, "messages": messages,
                "stream": False}
        data = self.transport(
            self.base_url + "/chat/completions", body)
        return parse_response(data)

## Step 5: Inspect a scripted response first

The fixture is a Python lookup with the same two-argument interface as the HTTP transport. It selects the last message and returns a compatible payload. It cannot learn, infer, obey prompts, or verify facts, and its usage counts are invented. The assertion checks a software expectation: the request and parsing path works for these fixtures. It says nothing about real-model accuracy or prompt compliance.

In [5]:
def fixture_transport(url, body):
    scripted = {"What is 2 + 2?": "4", "What is 3 + 5?": "8"}
    task = body["messages"][-1]["content"]
    answer = scripted.get(task, "UNSCRIPTED: connect a real model.")
    return {
        "model": "SCRIPTED-FIXTURE",
        "choices": [{"message": {"role": "assistant",
                                 "content": answer}}],
        "usage": {"prompt_tokens": 20, "completion_tokens": 1},
    }

llm = LLM("classroom-fixture", transport=fixture_transport)
response = llm.generate(messages)
print(asdict(response))
assert response.content == "4"

{'content': '4', 'reasoning': None, 'tool_call': None, 'metadata': {'model': 'SCRIPTED-FIXTURE', 'usage': {'prompt_tokens': 20, 'completion_tokens': 1}}}


## Step 6: Steps and trajectories

A `Response` represents one model call. A `Step` can record an answer, or a future action and observation. A `Trajectory` groups steps by task so failures can be located later. `add` refuses tool proposals with an explicit error because this answer-only harness cannot execute them, and it raises if no run has been initialized. A stored trajectory is not automatically conversation memory.

In [6]:
@dataclass
class Step:
    thought: str = ""
    action: dict | None = None
    observation: str | None = None
    answer: str | None = None
    metadata: dict = field(default_factory=dict)

class Trajectory:
    def __init__(self):
        self.runs = []

    def initialize(self, query):
        self.runs.append({"query": query, "steps": []})

    def add(self, response):
        if not self.runs:
            raise RuntimeError("Initialize a run first")
        if response.tool_call is not None:
            raise NotImplementedError("Tools are a later lesson")
        step = Step(answer=response.content,
                    metadata=dict(response.metadata))
        self.runs[-1]["steps"].append(step)

## Step 7: Connect TinyAgent

TinyAgent owns an LLM and a Trajectory. `run` starts a log, sends fresh messages, records the response, and returns the text. This is a single-step harness with no autonomous loop, and the next run will not remember this task because `run` never supplies the stored trajectory as context.

In [7]:
class TinyAgent:
    def __init__(self, llm):
        self.llm = llm
        self.trajectory = Trajectory()

    def run(self, task):
        self.trajectory.initialize(task)
        messages = [
            {"role": "system", "content": "Answer briefly and accurately."},
            {"role": "user", "content": task},
        ]
        response = self.llm.generate(messages)
        self.trajectory.add(response)
        return response.content

agent = TinyAgent(llm)
print(agent.run("What is 2 + 2?"))
print(agent.trajectory.runs)
assert len(agent.trajectory.runs) == 1
assert agent.trajectory.runs[0]["steps"][0].answer == "4"

4
[{'query': 'What is 2 + 2?', 'steps': [Step(thought='', action=None, observation=None, answer='4', metadata={'model': 'SCRIPTED-FIXTURE', 'usage': {'prompt_tokens': 20, 'completion_tokens': 1}})]}]


## Step 8: Controlled change and boundary checks

The task changes while the wrapper stays fixed. A second known task runs through the same software path, and assertions inspect both runs. The `try`/`except` block shows the expected bad-payload failure without stopping the notebook, and an unlisted question exposes the fixture boundary: the lookup has no answer for it.

In [8]:
print(agent.run("What is 3 + 5?"))
assert len(agent.trajectory.runs) == 2
assert all(len(run["steps"]) == 1 for run in agent.trajectory.runs)
assert agent.trajectory.runs[0]["steps"][0].answer == "4"
assert agent.trajectory.runs[1]["steps"][0].answer == "8"

try:
    parse_response({"choices": []})
except ValueError as error:
    print("Expected boundary error:", error)
else:
    raise AssertionError("Empty choices should fail")

print("Unlisted task:", agent.run("Name a planet."))

8
Expected boundary error: Endpoint returned no choices
Unlisted task: UNSCRIPTED: connect a real model.


## Investigation: add a fixture entry without changing the parser or agent

The lookup below adds one new question-answer pair. The parser, the LLM wrapper, and TinyAgent are reused exactly as written. The earlier runs stay unchanged, which the assertions check, and the new entry returns its stored value. An unlisted question still returns UNSCRIPTED. This shows the lookup only returns what was written into it; adding entries does not teach the fixture anything, and it is not evidence of generalization.

In [9]:
def fixture_transport_v2(url, body):
    scripted = {"What is 2 + 2?": "4", "What is 3 + 5?": "8",
                "What is 7 + 1?": "8"}
    task = body["messages"][-1]["content"]
    answer = scripted.get(task, "UNSCRIPTED: connect a real model.")
    return {
        "model": "SCRIPTED-FIXTURE",
        "choices": [{"message": {"role": "assistant",
                                 "content": answer}}],
        "usage": {"prompt_tokens": 20, "completion_tokens": 1},
    }

agent2 = TinyAgent(LLM("classroom-fixture", transport=fixture_transport_v2))
print(agent2.run("What is 7 + 1?"))
print("Unlisted task:", agent2.run("Name a planet."))
assert agent.trajectory.runs[0]["steps"][0].answer == "4"
assert agent.trajectory.runs[1]["steps"][0].answer == "8"
print("Earlier runs unchanged:", [run["steps"][0].answer for run in agent.trajectory.runs])

8
Unlisted task: UNSCRIPTED: connect a real model.
Earlier runs unchanged: ['4', '8', 'UNSCRIPTED: connect a real model.']


## Optional live extension (stays offline by default)

With `RUN_LIVE = False`, this cell does nothing. Setting it to `True` requires the one-time Ollama setup: Ollama installed, `gemma3:1b` pulled, and the server running on the same computer. A live run must be checked independently: verify the arithmetic yourself and record the actual model identity and usage if present. One correct live answer is very limited evidence of capability, and live output can vary.

In [10]:
# Optional extension. Default execution stays offline.
RUN_LIVE = False
MODEL = "gemma3:1b"  # Installed during the one-time setup.
if RUN_LIVE:
    if not MODEL:
        raise ValueError("Set MODEL to an installed local model tag")
    live_agent = TinyAgent(LLM(MODEL))
    print(live_agent.run("What is 2 + 2?"))
    print(live_agent.trajectory.runs)

## Debrief and exit ticket

**Call path.** `TinyAgent.run` builds the messages and calls `LLM.generate`, which builds the request body and calls the transport. The transport returns endpoint JSON, `parse_response` turns it into a `Response`, and `Trajectory.add` records it as a `Step`. The network boundary is the transport call; the parsing boundary is `parse_response`.

**Why logging is not memory.** The trajectory stores what happened, but `run` builds fresh messages every time and never includes the stored history. The model only sees what is in the messages it is given, so nothing is remembered unless the program puts it back in.

**Context budget.** 4096 - 3000 - 500 = 596 tokens remain under the toy assumptions. History, tool descriptions, retrieved text, and formatting overhead could all consume that space, and real endpoint limits vary.

**Fixture tradeoff.** The scripted fixture gives repeatable software checks without downloads, weights, or a server, but it is not model evidence. A lookup returning "4" says the plumbing works; it says nothing about accuracy, tokenization, latency, or prompt effectiveness. Real-model evidence needs a live request, the returned model identity and usage when available, and an answer checked independently.

**Training vs. inference.** Pre-training learns next-token patterns, SFT uses desired responses, and RL uses rewards. Prompting changes the input rather than the weights. Nothing in this notebook trains anything: the fixture lookup and the adapter only move and parse text.

**KV-cache tradeoff.** KV caching reuses stored keys and values instead of recomputing them, which saves computation at the cost of memory.